# **California Housing Regression**

**Goal:** Compare 8+ neural network configurations across architectures, loss functions,
and optimizers to find the best regressor for California median house prices.

## 1. Imports

In [ ]:
import os
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

# Reproducibility — seed all RNGs including GPU
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

os.makedirs('Plots', exist_ok=True)
os.makedirs('Weights', exist_ok=True)

## 2. Data Loading and Preprocessing

We will use 70% of the data for training, 15% for validation and 15% for testing.

In [ ]:
data = fetch_california_housing()
X, y = data.data, data.target.reshape(-1, 1)

print(f'Dataset shape  : X_Shape: {X.shape},  y_Shape: {y.shape}')
print(f'Feature names  : {list(data.feature_names)}')
print(f'Target range   : [{y.min():.2f}, {y.max():.2f}] (median house value, $100k units)')

# 3-way split: 70% train, 15% val, 15% test
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.15, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.15/0.85, random_state=SEED)

print(f'\nSplit sizes: train: {len(X_train)}, val: {len(X_val)}, test: {len(X_test)}')

# Fit scalers only on train set to avoid data leakage
scaler_X = StandardScaler().fit(X_train)
scaler_y = StandardScaler().fit(y_train)

X_train_s = scaler_X.transform(X_train)
X_val_s   = scaler_X.transform(X_val)
X_test_s  = scaler_X.transform(X_test)

y_train_s = scaler_y.transform(y_train)  # fixed: was fit_transform (redundant re-fit)
y_val_s   = scaler_y.transform(y_val)
y_test_s  = scaler_y.transform(y_test)


def make_loader(X_arr, y_arr, batch_size=64, shuffle=True):
    X_t = torch.tensor(X_arr, dtype=torch.float32)
    y_t = torch.tensor(y_arr, dtype=torch.float32)
    return DataLoader(TensorDataset(X_t, y_t), batch_size=batch_size, shuffle=shuffle)

## 3. Model architecture

In [ ]:
class RegressionNN(nn.Module):
    def __init__(self, input_dim: int, hidden_sizes: list, dropout_p: float = 0.0):
        super().__init__()
        if not hidden_sizes:
            raise ValueError('hidden_sizes must be a non-empty list')
        layers = []
        prev = input_dim
        for h in hidden_sizes:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            if dropout_p > 0.0:
                layers.append(nn.Dropout(p=dropout_p))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


demo = RegressionNN(input_dim=8, hidden_sizes=[64, 32], dropout_p=0.2)
print(demo)
print('Output shape:', demo(torch.randn(4, 8)).shape)

## 4. Experiment configuration

In [ ]:
architectures = {
    'A': [16, 8],
    'B': [64],
    'C': [128, 64, 32],
}

loss_funcs = {
    'MSE':      nn.MSELoss,
    'SmoothL1': nn.SmoothL1Loss,
}

# SGD now includes momentum=0.9 as stated in the README
optimizers = {
    'Adam': {'lr': 1e-3},
    'SGD':  {'lr': 1e-2, 'momentum': 0.9},
}

EPOCHS              = 80
BATCH_SIZE          = 64
WEIGHT_DECAY        = 1e-4
DROPOUT_P           = 0.0
SCHEDULER_PATIENCE  = 10   # halve LR after N epochs without val improvement
EARLY_STOP_PATIENCE = 20   # stop if no improvement for N epochs

## Training

In [ ]:
from copy import deepcopy

results    = []
all_curves = {}

# Build loaders once — data is identical across all runs
train_loader = make_loader(X_train_s, y_train_s, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = make_loader(X_val_s,   y_val_s,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = make_loader(X_test_s,  y_test_s,  batch_size=BATCH_SIZE, shuffle=False)

for arch_name, hidden in architectures.items():
    for loss_name, LossClass in loss_funcs.items():
        for opt_name, opt_kwargs in optimizers.items():

            run_name = f'{arch_name}_{hidden}_loss{loss_name}_opt{opt_name}'
            print(f'\n> {run_name}')

            torch.manual_seed(SEED)
            model     = RegressionNN(X_train_s.shape[1], hidden, dropout_p=DROPOUT_P).to(device)
            criterion = LossClass()

            if opt_name == 'Adam':
                optimizer = optim.Adam(model.parameters(), weight_decay=WEIGHT_DECAY, **opt_kwargs)
            else:
                optimizer = optim.SGD(model.parameters(), weight_decay=WEIGHT_DECAY, **opt_kwargs)

            scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=SCHEDULER_PATIENCE)

            train_losses, val_losses = [], []
            best_val_loss     = float('inf')
            best_weights      = None
            epochs_no_improve = 0

            for ep in tqdm(range(EPOCHS), desc=run_name, leave=False):

                # Training pass
                model.train()
                t_loss, t_n = 0.0, 0
                for xb, yb in train_loader:
                    xb, yb = xb.to(device), yb.to(device)
                    optimizer.zero_grad()
                    loss = criterion(model(xb), yb)
                    loss.backward()
                    optimizer.step()
                    t_loss += loss.item() * xb.size(0)
                    t_n    += xb.size(0)

                # Validation pass
                model.eval()
                v_loss, v_n = 0.0, 0
                with torch.no_grad():
                    for xb, yb in val_loader:
                        xb, yb = xb.to(device), yb.to(device)
                        v_loss += criterion(model(xb), yb).item() * xb.size(0)
                        v_n    += xb.size(0)

                avg_train = t_loss / t_n
                avg_val   = v_loss / v_n
                train_losses.append(avg_train)
                val_losses.append(avg_val)

                if avg_val < best_val_loss:
                    best_val_loss     = avg_val
                    best_weights      = deepcopy(model.state_dict())
                    epochs_no_improve = 0
                else:
                    epochs_no_improve += 1

                scheduler.step(avg_val)

                if epochs_no_improve >= EARLY_STOP_PATIENCE:
                    print(f'   Early stop at epoch {ep + 1}')
                    break

            weight_path = os.path.join('Weights', f'{run_name}.pt')
            torch.save(best_weights, weight_path)

            model.load_state_dict(best_weights)
            model.eval()
            preds_s, trues_s = [], []
            with torch.no_grad():
                for xb, yb in test_loader:
                    preds_s.append(model(xb.to(device)).cpu().numpy())
                    trues_s.append(yb.numpy())

            preds = scaler_y.inverse_transform(np.vstack(preds_s))
            trues = scaler_y.inverse_transform(np.vstack(trues_s))

            mse = mean_squared_error(trues, preds)
            r2  = r2_score(trues, preds)
            print(f'   Test MSE={mse:.4f}  R2={r2:.4f}')

            results.append({
                'run':           run_name,
                'architecture':  str(hidden),
                'loss_fn':       loss_name,
                'optimizer':     opt_name,
                'mse':           round(float(mse), 5),
                'r2':            round(float(r2),  5),
                'best_val_loss': round(float(best_val_loss), 6),
                'weight_file':   weight_path,
            })
            all_curves[run_name] = {'train': train_losses, 'val': val_losses}

            plt.figure(figsize=(6, 3))
            plt.plot(train_losses, label='Train loss')
            plt.plot(val_losses,   label='Val loss', linestyle='--')
            plt.title(f'Loss curves: {run_name}')
            plt.xlabel('Epoch')
            plt.ylabel('Loss')
            plt.legend()
            plt.grid(True)
            plt.tight_layout()
            plt.savefig(os.path.join('Plots', f'{run_name}_loss.png'), dpi=100)
            plt.close()

print('All runs finished.')

## 6. Results summary

In [ ]:
df = pd.DataFrame(results).sort_values('mse').reset_index(drop=True)
df.to_csv('Results_summary.csv', index=False)
df.style.highlight_min(subset=['mse'], color='lightgreen') \
        .highlight_max(subset=['r2'],  color='lightgreen')

## 7. Bar charts: MSE and R2 Across Experiments

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].barh(df['run'], df['mse'], color='steelblue')
axes[0].set_xlabel('Test MSE (lower is better)')
axes[0].set_title('Test MSE by Experiment')
axes[0].invert_yaxis()

axes[1].barh(df['run'], df['r2'], color='darkorange')
axes[1].set_xlabel('Test R2 (higher is better)')
axes[1].set_title('Test R2 by Experiment')
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig('Plots/metrics_comparison.png', dpi=120)
plt.show()

## 8. Best Model: Prediction vs True + Residuals

In [ ]:
import json

best_row = df.iloc[0]
best_run = best_row['run']
print(f'Best run: {best_run}  |  MSE={best_row["mse"]:.5f}  R2={best_row["r2"]:.5f}')

hidden     = json.loads(best_row['architecture'])
model_best = RegressionNN(X_test_s.shape[1], hidden, dropout_p=DROPOUT_P).to(device)
model_best.load_state_dict(torch.load(best_row['weight_file'], map_location=device))
model_best.eval()

preds_s, trues_s = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        preds_s.append(model_best(xb.to(device)).cpu().numpy())
        trues_s.append(yb.numpy())

preds     = scaler_y.inverse_transform(np.vstack(preds_s)).ravel()
trues     = scaler_y.inverse_transform(np.vstack(trues_s)).ravel()
residuals = trues - preds

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(trues, preds, s=8, alpha=0.4, color='steelblue')
lo, hi = min(trues.min(), preds.min()), max(trues.max(), preds.max())
axes[0].plot([lo, hi], [lo, hi], 'r--', lw=1.5, label='Perfect prediction')
axes[0].set_xlabel('True value ($100k)')
axes[0].set_ylabel('Predicted value ($100k)')
axes[0].set_title(f'Pred vs True: {best_run}')
axes[0].legend()

axes[1].scatter(preds, residuals, s=8, alpha=0.4, color='darkorange')
axes[1].axhline(0, color='red', linestyle='--', lw=1.5)
axes[1].set_xlabel('Predicted value ($100k)')
axes[1].set_ylabel('Residual (True - Predicted)')
axes[1].set_title('Residuals: Ideally scattered around 0')

plt.tight_layout()
plt.savefig('Plots/best_model_diagnostics.png', dpi=120)
plt.show()

## 9. Loss Curves

In [ ]:
n    = len(all_curves)
cols = 2
rows = (n + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(14, rows * 3))
axes = axes.ravel()

for i, (name, curves) in enumerate(all_curves.items()):
    axes[i].plot(curves['train'], label='Train')
    axes[i].plot(curves['val'],   label='Val', linestyle='--')
    axes[i].set_title(name, fontsize=9)
    axes[i].set_xlabel('Epoch')
    axes[i].set_ylabel('Loss')
    axes[i].legend(fontsize=8)
    axes[i].grid(True)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('All Loss Curves', fontsize=13)
plt.tight_layout()
plt.savefig('Plots/all_loss_curves.png', dpi=100)
plt.show()

## 10. Conclusions

1. All configurations are stable and capable of learning meaningful patterns from the data.
2. SmoothL1 provides a slight but consistent improvement over MSE when paired with Adam.
3. Adam converges faster and to a better solution than SGD with momentum across all architectures.
4. Deeper networks [128, 64, 32] outperform shallower ones — depth matters more than width.
5. Early stopping prevents wasted computation; most gains happen in the first 20-30 epochs.